In [49]:
import os
import csv
import re

## ==== USER DEFINED VARIABLES ===
main_path = "/Users/emluu/Documents/Siegel lab/machine learning/Alphafold3/phosphatases/translating_constraints/"
template_file = 'P6P_cyclic.enzdes.cst'
template_ligand = 'Pscicose_6P'
mapping_filename = 'atom_map.csv'

def load_csv_mapping(csv_path):
    mapping = {}
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            atoms = row['atoms'].strip()
            if atoms:  # Only include if atoms is not empty
                mapping[row['ligand']] = {
                    'residue_name': row['residue_name'],
                    'atoms': atoms.split()
                }
    return mapping

def replace_specific_atoms(atom_line, template_atoms, new_atoms):
    # Parse the atoms from the atom_name line in the block
    # atom_line example: 'TEMPLATE::   ATOM_MAP: 1 atom_name: P O4 C4'
    # Extract the atom names after 'atom_name:'
    prefix_match = re.match(r'^(.*atom_name:\s*)(.*)', atom_line)
    if not prefix_match:
        return atom_line  # If no match, return original line
    
    prefix = prefix_match.group(1)
    template_atom_list = prefix_match.group(2).split()

    # Replace template atoms only if mapped; preserve atom order limited to available mappings
    replaced_atoms = []
    for atom in template_atom_list:
        current_index = template_atoms.index(atom)
        replaced_atoms.append(new_atoms[current_index])
    
    return prefix + " ".join(replaced_atoms)

def rewrite_block(template_block, old_ligand, old_atoms, new_residue, new_atoms):
    lines = template_block.split('\n')
    new_lines = []
    for idx, line in enumerate(lines):

        # Replace residue3 field as before
        if f'residue3: {old_ligand}' in line:
            line = line.replace(f'residue3: {old_ligand}', f'residue3: {new_residue}')
            new_lines[idx-1]=replace_specific_atoms(new_lines[idx-1], old_atoms, new_atoms)

        new_lines.append(line)
    return '\n'.join(new_lines)

def process_template(template_path,template_resiname, template_atoms,mapping, output_dir):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    with open(template_path) as f:
        template = f.read()
    
    blocks = template.split('CST::END')
    blocks = [b for b in blocks if b.strip()]

    for ligand, data in mapping.items():
        new_blocks = []
        for block in blocks:
            if template_resiname in block:
                new_block = rewrite_block(block, template_resiname,template_atoms, data['residue_name'], data['atoms'])
            else:
                new_block = block
            if new_block.strip():
                new_blocks.append(new_block.strip() + "\nCST::END\n")
        
        outfile_path = os.path.join(output_dir, f"{ligand}.enzdes.cst")
        with open(outfile_path, 'w') as out:
            out.writelines(new_blocks)
        print(f"Wrote: {outfile_path}")

# Example usage:
mapping = load_csv_mapping(os.path.join(main_path,mapping_filename))

process_template(os.path.join(main_path,template_file),
                mapping[template_ligand]['residue_name'],
                mapping[template_ligand]['atoms'],
                mapping, 
                os.path.join(main_path,'output_constraints'))


Wrote: /Users/emluu/Documents/Siegel lab/machine learning/Alphafold3/phosphatases/translating_constraints/output_constraints/Pscicose_6P.enzdes.cst
Wrote: /Users/emluu/Documents/Siegel lab/machine learning/Alphafold3/phosphatases/translating_constraints/output_constraints/Sedoheptulose_7P.enzdes.cst
Wrote: /Users/emluu/Documents/Siegel lab/machine learning/Alphafold3/phosphatases/translating_constraints/output_constraints/D-Mannose_6P.enzdes.cst
Wrote: /Users/emluu/Documents/Siegel lab/machine learning/Alphafold3/phosphatases/translating_constraints/output_constraints/D-Glucose_1P.enzdes.cst
Wrote: /Users/emluu/Documents/Siegel lab/machine learning/Alphafold3/phosphatases/translating_constraints/output_constraints/D-Sorbitol_6P.enzdes.cst
Wrote: /Users/emluu/Documents/Siegel lab/machine learning/Alphafold3/phosphatases/translating_constraints/output_constraints/D-Erythrose_4P.enzdes.cst
Wrote: /Users/emluu/Documents/Siegel lab/machine learning/Alphafold3/phosphatases/translating_constr